# 02 · Feature Engineering
### Construct lag, rolling, shock, and trend features for modeling

**Author:** Ibrahim Denis Fofanah — Pace University | RiseAfrica Foundation for STEM and Innovation
**Project:** Machine Learning Approaches to Crop Yield Prediction and Post-Harvest Loss Reduction — Evidence from Sierra Leone

---

In [1]:
# ── Setup & project-root anchoring ────────────────────────────────────────────
import sys, os, warnings

# Resolve paths whether launched from repo root or from notebooks/
if not os.path.exists('data/raw') and os.path.exists('../data/raw'):
    os.chdir('..')
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 50)

for d in ['data/processed', 'outputs/figures', 'outputs/models', 'outputs/reports']:
    os.makedirs(d, exist_ok=True)

print(f'Working directory: {os.getcwd()}')

Working directory: /Users/ibrahimfofanah/sierraleone-agri-ml


## 1. Load Analysis Dataset

In [2]:
analysis = pd.read_csv('data/processed/analysis_dataset.csv')
print(f'Loaded: {analysis.shape[0]} rows x {analysis.shape[1]} columns')
analysis.head()

Loaded: 25 rows x 135 columns


,Year,"Broad beans and horse beans, dry_Yield",Cabbages_Yield,"Cashew nuts, in shell_Yield",Cassava_Yield,Cereals n.e.c._Yield,"Cereals, primary_Yield","Chillies and peppers, dry (Capsicum spp., Pimenta spp.), raw_Yield","Chillies and peppers, green (Capsicum spp. and Pimenta spp.)_Yield",Cocoa beans_Yield,"Coconuts, in shell_Yield","Coffee, green_Yield",Cucumbers and gherkins_Yield,Eggplants (aubergines)_Yield,Fruit Primary_Yield,Groundnuts_Yield,Kola nuts_Yield,"Lentils, dry_Yield",Maize_Yield,"Mangoes, guavas and mangosteens_Yield",Millet_Yield,Oil palm fruit_Yield,Okra_Yield,"Other citrus fruit, n.e.c._Yield","Other fibre crops, raw, n.e.c._Yield",...,Maize_Area,"Mangoes, guavas and mangosteens_Area",Millet_Area,Oil palm fruit_Area,Okra_Area,"Other citrus fruit, n.e.c._Area","Other fibre crops, raw, n.e.c._Area","Other fruits, n.e.c._Area",Other pulses n.e.c._Area,"Other stimulant, spice and aromatic crops, n.e.c._Area","Other vegetables, fresh n.e.c._Area","Peas, dry_Area",Plantains_Area,"Pumpkins, squash and gourds_Area",Rice_Area,Sesame seed_Area,Sorghum_Area,Soya beans_Area,Sugar cane_Area,Sweet Potato_Area,Taro_Area,Tea leaves_Area,Tomatoes_Area,Unmanufactured tobacco_Area,Yams_Area
0,2000,1200.00,NaN,NaN,6762.70,818.20,1078.20,2517.50,NaN,300.00,957.90,1666.70,NaN,NaN,4694.90,772.80,277.80,NaN,928.50,1666.70,908.30,8027.50,NaN,5694.50,5188.40,...,9588.00,4200.00,4003.00,21800.00,NaN,14141.00,1000.00,12500.00,60000.00,2003.00,26600.00,1275.00,5482.00,NaN,183214.00,3600.00,6231.00,NaN,1150.00,10642.00,960.00,NaN,1209.00,40.00,NaN
1,2001,1250.00,NaN,NaN,6266.00,794.90,998.20,2447.20,NaN,275.00,963.00,1684.10,NaN,NaN,4929.20,769.20,289.50,NaN,1030.90,1818.20,885.40,8181.80,NaN,5693.50,4794.30,...,9700.00,4400.00,9600.00,22000.00,NaN,14401.00,1200.00,11000.00,64000.00,2316.00,32000.00,1400.00,5700.00,NaN,300000.00,3100.00,12000.00,NaN,1000.00,10000.00,1000.00,NaN,1350.00,36.00,NaN
2,2002,1200.00,NaN,NaN,6166.20,785.70,995.90,2776.50,NaN,319.40,965.70,1674.80,NaN,NaN,4901.60,768.40,290.00,NaN,787.00,2222.20,906.70,8181.80,NaN,5694.50,4974.30,...,15297.00,4500.00,11612.00,22000.00,NaN,14683.00,1250.00,12000.00,70000.00,2648.00,33000.00,1500.00,5971.00,NaN,420000.00,2900.00,17298.00,NaN,1000.00,10319.00,1052.00,NaN,1500.00,42.00,NaN
3,2003,1227.30,NaN,NaN,6229.60,790.70,1011.80,2990.70,NaN,363.60,965.50,1700.00,NaN,NaN,4947.50,766.30,285.70,NaN,1003.80,2441.90,1000.00,8125.00,NaN,5666.70,4840.70,...,16000.00,4300.00,10000.00,24000.00,NaN,15000.00,1400.00,12500.00,82000.00,2996.00,35000.00,1550.00,6300.00,NaN,440000.00,3300.00,20000.00,NaN,1000.00,48297.00,1050.00,NaN,1500.00,50.00,NaN
4,2004,1200.00,NaN,NaN,6283.60,795.50,1000.30,2494.50,NaN,375.00,966.70,1636.40,NaN,NaN,4878.00,759.40,295.20,NaN,973.50,2452.80,1000.00,8076.90,NaN,5612.90,5000.00,...,33000.00,5300.00,15000.00,26000.00,NaN,15500.00,1500.00,13500.00,90000.00,3360.00,39000.00,1750.00,6700.00,NaN,540000.00,3600.00,21605.00,NaN,1000.00,48092.00,1080.00,NaN,1800.00,55.00,NaN


## 2. Engineer Temporal, Lag & Shock Features

In [3]:
from src import feature_engineering as fe

ml_df = fe.add_temporal_features(analysis)
print(f'Processed dataset: {ml_df.shape[0]} rows x {ml_df.shape[1]} columns')
new_cols = [c for c in ml_df.columns if c not in analysis.columns]
print(f'\nNew feature columns ({len(new_cols)}):')
print(new_cols)

[✓] Feature engineering: 153 columns (18 new features)
Processed dataset: 25 rows x 153 columns

New feature columns (18):
['Rice_Yield_Lag1', 'Rice_Yield_Lag2', 'Rice_Yield_Lag3', 'Rice_Yield_Rolling3', 'Rice_Yield_Rolling5', 'Rice_Yield_YoY', 'Cassava_Yield_Lag1', 'Cassava_YoY', 'Rice_Area_Lag1', 'Rice_Area_Change', 'Rice_Prod_Per_Ha', 'Year_Trend', 'Decade', 'Ebola_Period', 'COVID_Period', 'FeedSalone', 'Post2010', 'Crop_Diversity_Yield']


## 3. Save Processed Dataset

In [4]:
ml_df.to_csv('data/processed/processed_dataset.csv', index=False)
print('[saved] data/processed/processed_dataset.csv')
ml_df[['Year', 'Rice_Yield', 'Rice_Yield_Lag1', 'Rice_Yield_Rolling3',
       'Rice_Yield_YoY', 'Ebola_Period', 'COVID_Period', 'FeedSalone']].tail(10)

[saved] data/processed/processed_dataset.csv


,Year,Rice_Yield,Rice_Yield_Lag1,Rice_Yield_Rolling3,Rice_Yield_YoY,Ebola_Period,COVID_Period,FeedSalone
15,2015,1441.00,1623.40,1644.80,-11.24,1,0,0
16,2016,1143.70,1441.00,1402.70,-20.63,1,0,0
17,2017,1149.20,1143.70,1244.63,0.48,0,0,0
18,2018,785.60,1149.20,1026.17,-31.64,0,0,0
19,2019,1574.50,785.60,1169.77,100.42,0,0,0
20,2020,1736.90,1574.50,1365.67,10.31,0,1,0
21,2021,2095.30,1736.90,1802.23,20.63,0,1,0
22,2022,2109.80,2095.30,1980.67,0.69,0,0,0
23,2023,1508.50,2109.80,1904.53,-28.50,0,0,1
24,2024,1408.60,1508.50,1675.63,-6.62,0,0,1
